In [2]:
#!/usr/bin/env python3
"""
check_eval_raw_counts.py

Counts questions per file in data/eval/raw/*.json and prints totals + type breakdowns.
Assumes BioASQ-like JSON with a top-level "questions" list (golden/training).
"""

import json
from pathlib import Path
from glob import glob
from collections import Counter, defaultdict
import sys

SRC_DIR = Path("../data/eval/raw")

def main():
    if not SRC_DIR.exists():
        print(f"❌ Directory not found: {SRC_DIR}")
        sys.exit(1)

    files = sorted(glob(str(SRC_DIR / "*.json")))
    if not files:
        print(f"❌ No .json files found in {SRC_DIR}")
        sys.exit(1)

    grand_total = 0
    grand_types = Counter()
    per_file_types = defaultdict(Counter)

    print(f"📂 Found {len(files)} files in {SRC_DIR}\n")

    for fp in files:
        p = Path(fp)
        try:
            with open(p, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            print(f"⚠️  Skipping {p.name}: failed to parse JSON ({e})")
            continue

        questions = data.get("questions", [])
        q_count = len(questions)
        grand_total += q_count

        # Per-file type counts
        for q in questions:
            qtype = (q.get("type") or q.get("questionType") or "unknown").lower()
            per_file_types[p.name][qtype] += 1
            grand_types[qtype] += 1

        # Label files as training/golden by filename
        label = (
            "training" if "train" in p.stem.lower()
            else ("golden" if "gold" in p.stem.lower() else "unknown")
        )

        print(f"{p.name:<25}  [{label:8}]  questions: {q_count}")
        if q_count:
            breakdown = ", ".join(f"{t}:{c}" for t, c in sorted(per_file_types[p.name].items()))
            print(f"  types: {breakdown}")
        print()

    print("=== Totals ===")
    print(f"All files questions: {grand_total}")
    if grand_types:
        print("By type:")
        for t, c in sorted(grand_types.items()):
            print(f"  {t:<8} {c}")

if __name__ == "__main__":
    main()


📂 Found 9 files in ../data/eval/raw

11B1_golden.json           [golden  ]  questions: 75
  types: factoid:19, list:12, summary:20, yesno:24

11B2_golden.json           [golden  ]  questions: 75
  types: factoid:22, list:12, summary:17, yesno:24

11B3_golden.json           [golden  ]  questions: 90
  types: factoid:26, list:18, summary:22, yesno:24

11B4_golden.json           [golden  ]  questions: 90
  types: factoid:31, list:24, summary:21, yesno:14

12B1_golden.json           [golden  ]  questions: 85
  types: factoid:21, list:21, summary:18, yesno:25

12B2_golden.json           [golden  ]  questions: 85
  types: factoid:19, list:18, summary:22, yesno:26

12B3_golden.json           [golden  ]  questions: 85
  types: factoid:26, list:19, summary:16, yesno:24

12B4_golden.json           [golden  ]  questions: 85
  types: factoid:19, list:22, summary:17, yesno:27

training13b.json           [training]  questions: 5389
  types: factoid:1600, list:1047, summary:1283, yesno:1459

=== Tota